# PINN Cohort Training on GPU

Run the per-transition PINN forecaster on Google Colab's free T4 GPU.

**Setup:**
1. Upload your data to Google Drive under `My Drive/STS_Data/`
2. Required files in that folder:
   - `mu_glioma_post/` (raw NIfTI directory)
   - `shared_model_selection_transitions.json`
   - `shared_training_transitions.json`
   - `shared_split_manifest.json`

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone the repo and install
!git clone https://github.com/tanushappapogu-max/BINNs.git /content/BINNs
%cd /content/BINNs
!pip install -e '.[ml,imaging]' -q

In [ ]:
# Verify GPU availability
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
# Configure paths
from pathlib import Path

DRIVE_DATA = Path('/content/drive/MyDrive/STS_Data')
NIFTI_ROOT = DRIVE_DATA / 'mu_glioma_post'
MANIFEST = DRIVE_DATA / 'shared_split_manifest.json'

# Choose which split to run
ROLE = 'model_selection'  # 'model_selection' for validation (39), 'training' for full (208)
TRANSITION_INDEX = DRIVE_DATA / f'shared_{ROLE}_transitions.json'
OUTPUT_ROOT = Path(f'/content/drive/MyDrive/STS_Data/outputs/pinn_cohort_{ROLE}')

for p in [NIFTI_ROOT, MANIFEST, TRANSITION_INDEX]:
    assert p.exists(), f'Missing: {p}'

In [ ]:
import json
from gbm_pinn.pinn_cohort import PINNCohortConfig, run_pinn_cohort

config = PINNCohortConfig(
    transition_index_path=TRANSITION_INDEX,
    manifest_path=MANIFEST,
    nifti_root=NIFTI_ROOT,
    output_root=OUTPUT_ROOT,
    role=ROLE,
    device='cuda',
    epochs=2000,
    resume=True,
)

summary = run_pinn_cohort(config)
print(json.dumps(summary, indent=2))

In [ ]:
# Analyze results
import json
summary = json.load(open(OUTPUT_ROOT / 'cohort_summary.json'))
print(f"Mean Dice: {summary['mean_dice']:.4f}")
print(f"Mean Skill: {summary['mean_dice_skill_over_persistence']:+.4f}")
print(f"Beating Persistence: {summary['n_beating_persistence']}/{summary['n_successful']}")